# RepairWise Gemma — Ollama Local Demo
## Gemma 4 Good Hackathon · Special Technology Prize Track (Ollama)

Run RepairWise with **Gemma 4 E2B via Ollama** — fully local, no GPU required for inference,
no HuggingFace token needed after model download.

This notebook demonstrates:
- Installing and running Ollama in Kaggle
- Pulling `gemma4:e2b` model
- Running RepairWise multilingual phone repair assistant via Ollama
- Streaming responses token by token
- Multimodal photo analysis (SMS screenshots, physical damage)

### Why Ollama?
Ollama makes Gemma 4 accessible to anyone with a laptop — no Python GPU stack,
no CUDA drivers. A phone repair shop owner can run RepairWise locally with:
```bash
ollama run gemma4:e2b
python app.py
```


## 1. Install Ollama

In [1]:
# 1. Instalar zstd
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 207 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (13.5 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 120314 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
import subprocess, sys, os, time

# Install Ollama (Linux)
print("Installing Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.ai/install.sh | sh",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-200:] if result.stderr else "")
print("✅ Ollama installed")


Installing Ollama...

...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

✅ Ollama installed


## 2. Start Ollama server and pull Gemma 4 E2B

In [3]:
import subprocess, time, urllib.request, json

# Start Ollama server in background
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

# Verify server is running
try:
    resp = urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5)
    print("✅ Ollama server running")
except Exception as e:
    print(f"⚠️  Server may still be starting: {e}")

# Pull Gemma 4 E2B
print("Pulling gemma4:e2b (this may take a few minutes)...")
result = subprocess.run(
    ["ollama", "pull", "gemma4:e2b"],
    capture_output=True, text=True, timeout=600
)
print(result.stdout[-300:] if result.stdout else "")
print("✅ gemma4:e2b ready")


✅ Ollama server running
Pulling gemma4:e2b (this may take a few minutes)...

✅ gemma4:e2b ready


## 3. Test raw Ollama connection

In [4]:
import urllib.request, json

def ollama_generate(prompt: str, model: str = "gemma4:e2b", max_tokens: int = 200) -> str:
    payload = json.dumps({
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": max_tokens,
            "temperature": 0.4,
            "repeat_penalty": 1.12,
            "stop": ["<end_of_turn>", "<start_of_turn>"],
        }
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
        return data.get("response", "").strip()

# Quick sanity check
response = ollama_generate("Say hello in Spanish in one sentence.")
print("Ollama response:", response)


Ollama response: Hola.


## 4. Load RepairWise with Ollama backend

In [5]:
import os
import sys

# Point to our RepairWise files (copy from /kaggle/input or define inline)
os.environ["REPAIRWISE_BACKEND"] = "ollama"
os.environ["OLLAMA_URL"] = "http://localhost:11434"
os.environ["OLLAMA_MODEL"] = "gemma4:e2b"

# If running from Kaggle with our dataset attached:
# sys.path.insert(0, "/kaggle/input/repairwise-gemma")

# For this notebook we inline the core functions
# In production: from engine import repairwise_debug

print("✅ Environment configured for Ollama backend")
print(f"  REPAIRWISE_BACKEND: {os.environ['REPAIRWISE_BACKEND']}")
print(f"  OLLAMA_URL: {os.environ['OLLAMA_URL']}")
print(f"  OLLAMA_MODEL: {os.environ['OLLAMA_MODEL']}")


✅ Environment configured for Ollama backend
  REPAIRWISE_BACKEND: ollama
  OLLAMA_URL: http://localhost:11434
  OLLAMA_MODEL: gemma4:e2b


## 5. RepairWise Ollama — multilingual demo

In [6]:
import urllib.request, json

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma4:e2b"

SYSTEM_PROMPT = (
    "You are RepairWise Gemma, a professional phone repair technician and digital safety advisor. "
    "Help people — especially elderly users and immigrants — understand phone problems and avoid SMS scams. "
    "Always respond in the SAME language as the user. "
    "Use EXACTLY this 5-section structure:\n"
    "1. Probable diagnosis:\n"
    "2. Risk level: HIGH 🔴 / MEDIUM 🟡 / LOW 🟢\n"
    "3. What to do right now:\n"
    "4. When to see a professional:\n"
    "5. Sources used: [knowledge_id]"
)

def repairwise_ollama(user_query: str, language: str = "Spanish") -> str:
    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"Language: {language}\n"
        f"Customer query: {user_query}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": -1,
            "temperature": 0.4,
            "repeat_penalty": 1.12,
            "stop": ["<end_of_turn>", "<start_of_turn>"],
        }
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        data = json.loads(resp.read())
        return data.get("response", "").replace("<end_of_turn>","").strip()

test_cases = [
    ("Me ha llegado un SMS del banco con un link y me pide la tarjeta.", "Spanish"),
    ("Mi batería está hinchada y la pantalla se está levantando.", "Spanish"),
    ("I dropped my phone in water and now it is getting hot.", "English"),
    ("El meu mòbil no s'encén i es queda al logo.", "Catalan"),
    ("Telefonul meu nu se încarcă deloc.", "Romanian"),
    ("مجھے بینک کا ایک پیغام آیا ہے جس میں کارڈ کی تفصیلات مانگی گئی ہیں۔", "Urdu"),
]

print("=" * 70)
print("RepairWise Gemma — Ollama Backend (gemma4:e2b)")
print("=" * 70)

for query, lang in test_cases:
    print(f"\n[{lang}] {query[:60]}...")
    try:
        answer = repairwise_ollama(query, lang)
        print(answer if answer else "⚠️ Empty response")
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 70)

RepairWise Gemma — Ollama Backend (gemma4:e2b)

[Spanish] Me ha llegado un SMS del banco con un link y me pide la tarj...
1. Probable diagnosis:
Usted ha recibido un intento de estafa conocido como "smishing" (phishing por SMS). Los estafadores utilizan mensajes falsos para intentar engañarle y robar su información bancaria o datos personales. **No haga clic en ningún enlace ni proporcione su tarjeta.**

2. Risk level: HIGH 🔴

3. What to do right now:
*   **NO haga clic** en el enlace del SMS.
*   **NO introduzca** su número de tarjeta, PIN o cualquier otra información personal.
*   **NO responda** al mensaje.
*   Elimine el mensaje.
*   Si tiene dudas, contacte directamente a su banco por teléfono (usando el número oficial que tiene, no el que aparece en el SMS) para verificar si el mensaje es legítimo.

4. When to see a professional:
Si siente que ha proporcionado información o si el intento de estafa fue muy insistente, debe reportarlo inmediatamente a su banco y considerar reportar

## 6. Streaming demo — token by token

In [7]:
# Test one at a time with long timeout
def repairwise_ollama(user_query: str, language: str = "Spanish") -> str:
    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"Language: {language}\n"
        f"Customer query: {user_query}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": -1,
            "temperature": 0.4,
            "repeat_penalty": 1.12,
            "stop": ["<end_of_turn>", "<start_of_turn>"],
        }
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        data = json.loads(resp.read())
        return data.get("response", "").replace("<end_of_turn>","").strip()

# Test Spanish scam
print("[Spanish] SMS phishing:")
print(repairwise_ollama("Me ha llegado un SMS del banco con un link y me pide la tarjeta.", "Spanish"))

[Spanish] SMS phishing:
1. Probable diagnosis:
Usted ha recibido un intento de estafa conocido como "smishing" (phishing por mensaje de texto). Los estafadores utilizan mensajes falsos para intentar engañarle y obtener su información bancaria o datos personales a través de enlaces maliciosos.

2. Risk level: HIGH 🔴

3. What to do right now:
**NO haga clic en ningún enlace** dentro del mensaje. **NO introduzca ninguna información** como números de tarjeta, contraseñas o códigos de seguridad. **NO responda** al mensaje. Si tiene la tarjeta, no la envíe. Elimine el mensaje.

4. When to see a professional:
Si ya ha hecho clic en el enlace, ha introducido datos, o si siente mucha preocupación, debe contactar inmediatamente a su banco o a la policía local para reportar el intento de fraude.

5. Sources used: [knowledge_id]


In [8]:
import urllib.request, json

def repairwise_ollama_stream(user_query: str, language: str = "Spanish"):
    """Stream RepairWise response token by token via Ollama."""
    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\nLanguage: {language}\nCustomer query: {user_query}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": True,
        "options": {"num_predict": -1, "temperature": 0.4, "repeat_penalty": 1.12,
                    "stop": ["<end_of_turn>", "<start_of_turn>"]},
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    full = ""
    with urllib.request.urlopen(req, timeout=90) as resp:
        for line in resp:
            if line:
                chunk = json.loads(line)
                token = chunk.get("response", "")
                print(token, end="", flush=True)
                full += token
                if chunk.get("done"):
                    break
    print()
    return full

print("Streaming demo — Mi batería está hinchada:")
print("-" * 70)
result = repairwise_ollama_stream(
    "Mi batería está hinchada y la pantalla se levanta.",
    language="Spanish"
)


Streaming demo — Mi batería está hinchada:
----------------------------------------------------------------------
1. Probable diagnosis:
Esto indica un problema grave con la batería de su teléfono. La hinchazón de la batería (hinchazón) es una señal de que la batería está dañada internamente, lo que puede causar un riesgo de incendio. La pantalla que se levanta es un síntoma de la presión interna generada por esta hinchazón.

2. Risk level: HIGH 🔴

3. What to do right now:
**¡Deténgase inmediatamente!** No cargue el teléfono y no lo use. Apague el dispositivo de inmediato. No intente cargar la batería ni manipularla. Mantenga el teléfono en un lugar seguro, lejos de materiales inflamables (como papel o telas). No lo cargue.

4. When to see a professional:
Debe llevar el teléfono a un técnico profesional o a un centro de reparación lo antes posible. No intente abrir el teléfono usted mismo, ya que esto podría empeorar el peligro. La seguridad es lo más importante.

5. Sources used: [kno

## 7. Multimodal — photo analysis via Ollama

In [9]:
import urllib.request, json, base64
from PIL import Image, ImageDraw
from io import BytesIO

def create_sms_screenshot():
    """Create a synthetic phishing SMS screenshot for demo."""
    img = Image.new("RGB", (600, 280), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, 600, 50], fill="#CC0000")
    d.text((15, 12), "⚠️  BANCO SEGURO — ALERTA", fill="white")
    d.text((15, 70), "Su cuenta ha sido SUSPENDIDA por actividad inusual.", fill="black")
    d.text((15, 100), "Verifique AHORA: http://banco-seguro-verificar.example/login", fill="#CC0000")
    d.text((15, 130), "Introduzca: número de tarjeta + PIN + código OTP", fill="#880000")
    d.text((15, 180), "⚠️ AVISO: Los bancos reales NUNCA piden esto por SMS.", fill="#007700")
    return img

def analyse_photo_ollama(image: Image.Image, query: str, language: str = "Spanish") -> str:
    """Send image + text to Ollama for multimodal analysis."""
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    img_b64 = base64.b64encode(buf.getvalue()).decode()

    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\nLanguage: {language}\n"
        f"The user has uploaded an image. Analyse it carefully.\n"
        f"Customer query: {query}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "images": [img_b64],
        "stream": False,
        "options": {"num_predict": -1, "temperature": 0.4, "repeat_penalty": 1.12,
                    "stop": ["<end_of_turn>", "<start_of_turn>"]},
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        data = json.loads(resp.read())
        return data.get("response", "").replace("<end_of_turn>","").strip()

# Demo
sms_img = create_sms_screenshot()
print("Analysing SMS screenshot with Ollama vision...")
print("-" * 70)
answer = analyse_photo_ollama(sms_img, "¿Es real este mensaje del banco?", "Spanish")
print(answer)


Analysing SMS screenshot with Ollama vision...
----------------------------------------------------------------------
1. Probable diagnosis: El mensaje que usted recibió es una posible estafa de *phishing*. Los estafadores suelen enviar mensajes falsos haciéndose pasar por bancos o entidades oficiales para intentar robar su información personal, contraseñas o datos bancarios.
2. Risk level: HIGH 🔴
3. What to do right now: NO haga clic en ningún enlace ni proporcione ninguna información personal (contraseñas, números de tarjeta, códigos de seguridad) a través de ese mensaje. No responda al mensaje. Verifique la autenticidad del mensaje contactando directamente a su banco a través de sus canales oficiales (teléfono o página web oficial), no utilizando la información de contacto que aparece en el mensaje sospechoso.
4. When to see a professional: Si usted ha proporcionado información o ha hecho clic en algo por error, o si sospecha que ha sido víctima de un fraude, debe contactar inmediat

## 8. Architecture summary

```
User (local machine)
      │
      │  ollama run gemma4:e2b
      ▼
┌──────────────────────┐
│  Ollama Server       │  http://localhost:11434
│  gemma4:e2b          │  Text + Vision
│  Runs on CPU or GPU  │  No Python GPU stack needed
└──────────┬───────────┘
           │  HTTP API /api/generate
           ▼
┌──────────────────────┐
│  RepairWise Engine   │  engine.py
│  Safety Triage       │  Deterministic — always runs first
│  4-Engine RAG        │  TF-IDF + sentence-transformers
│  Ollama backend      │  REPAIRWISE_BACKEND=ollama
└──────────┬───────────┘
           │
           ▼
  Structured answer (6 languages)
  Diagnosis · Risk · Steps · Sources
```

### Running locally (anyone can do this)
```bash
# 1. Install Ollama
curl -fsSL https://ollama.ai/install.sh | sh

# 2. Pull Gemma 4
ollama pull gemma4:e2b

# 3. Clone RepairWise
git clone https://github.com/SocAbdul/repairwise-gemma
cd repairwise-gemma
pip install -r requirements.txt

# 4. Run with Ollama backend
REPAIRWISE_BACKEND=ollama python app.py
```

No HuggingFace token. No GPU required. Works on any laptop with 8GB RAM.
